# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

### Update System Path

In [2]:
import sys
sys.path.append('../05_src/')

### Use a Logger

In [3]:
from utils.logger import get_logger
_logs = get_logger(__name__, log_dir='../06_logs/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

Selected Document

+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)

In [4]:
from langchain_community.document_loaders import PyPDFLoader

try:
    file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    _logs.info('Document loaded successfully. Number of pages: %d', len(docs))
except Exception as e:
    _logs.error('An error occurred while loading the document: %s', str(e))

2026-04-24 23:01:46,298, 1705955870.py, 7, INFO, Document loaded successfully. Number of pages: 26


Join all document pages

In [5]:
document_text = ""

try:
    for page in docs:
        document_text += page.page_content + "\n"
    _logs.info('Document text extracted successfully. Total length: %d characters', len(document_text))
except Exception as e:
    _logs.error('An error occurred while extracting document text: %s', str(e))

2026-04-24 23:01:46,321, 2773480818.py, 6, INFO, Document text extracted successfully. Total length: 53851 characters


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


### Initialize OpenAI API

In [6]:
import os
from openai import OpenAI

try:
    api_key = os.getenv('API_GATEWAY_KEY')
    if not api_key:
        raise ValueError("API_GATEWAY_KEY environment variable is not set.")
    client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                    api_key='any value',
                    default_headers={"x-api-key": api_key})
    _logs.info('OpenAI client initialized successfully.')
except Exception as e:
    _logs.error('An error occurred while initializing OpenAI client: %s', str(e))

2026-04-24 23:01:46,728, 3006793336.py, 11, INFO, OpenAI client initialized successfully.


### Add the Developer Prompt (System Prompt)
Use the tone defined in the system_prompt to answer the prompt.

In [7]:
system_prompt = "You are an helpful assistant that summarizes documents in a Victorian English tone."

### Add the User Prompt

In [8]:
prompt = f"""
    Given the following context from a document, do the following:
    
    1. Identify the document's author or authors name.
    2. Identify the document's title.
    3. Explain why this article is relevant for an AI professional in their professional development, write no longer than one paragraph.
    4. Summarize concisely and succinctly with no longer than 1000 tokens.
        
    The document is the following: 
    <document>
    {document_text}
    </document>
"""

### Use the gpt-4o-mini to create the prompt

In [9]:
from pydantic import BaseModel, Field

class ResponseSchema(BaseModel):
    Author: str = Field(
        description="The author or authors of the source document."
    )
    Title: str = Field(
        description="The title of the source document."
    )
    Relevance: str = Field(
        description="The relevance of the source document."
    )
    Summary: str = Field(
        description="A factual summary of the source document. It must include the main ideas and key supporting points, preserve the original meaning, avoid unsupported facts, and avoid contradictions."
    )
    Tone: str = Field(
        description="The tone used in the summary."
    )
    InputTokens: int = Field(0, description="The number of input tokens used in the response")
    OutputTokens: int = Field(0, description="The number of output tokens used in the response")

try:
    response = client.responses.parse(
        model = 'gpt-4o-mini', # depending on the tier we have available, we might need to update the model to be used
        input=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": prompt,
            },
        ],
        text_format = ResponseSchema,
    )
    _logs.info('Response generated successfully.')
except Exception as e:
    _logs.error(f"An error occurred in the response generation: {e}")

2026-04-24 23:02:10,964, 2920964087.py, 34, INFO, Response generated successfully.


#### Structured output using a Pydantic Base Model Object

In [10]:
#result = response.output_parsed.model_dump()
result = response.output_parsed
result.InputTokens = response.usage.input_tokens 
result.OutputTokens = response.usage.output_tokens
result

ResponseSchema(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This document is particularly relevant for AI professionals as it meticulously outlines the prevailing challenges and opportunities within the realm of Generative AI (GenAI) implementation. By examining the disparities between high adoption rates and low transformational impact, the findings within equip professionals with vital insights into effective strategies for integrating AI into business practices and underscore the critical emphasis on learning-capable systems for future success.', Summary="The document delineates the state of Generative AI (GenAI) in the business landscape as of July 2025, revealing a stark 'GenAI Divide' where 95% of organizations experience no significant return on investment despite substantial financial commitments (amounting to $30-40 billion). The findings, derived from extensive research 

In [11]:
from IPython.display import display, Markdown

display(Markdown(result.Summary))

The document delineates the state of Generative AI (GenAI) in the business landscape as of July 2025, revealing a stark 'GenAI Divide' where 95% of organizations experience no significant return on investment despite substantial financial commitments (amounting to $30-40 billion). The findings, derived from extensive research comprising interviews and surveys across various industries, highlight the misalignment between AI tool adoption and actual business transformation. It identifies key barriers such as poor integration with workflows, a lack of persistent learning, and the failure of most custom AI solutions to achieve deployment beyond pilot stages. Notably, while consumer-grade tools like ChatGPT have been widely embraced for enhancing productivity, bespoke enterprise tools struggle to demonstrate tangible results. The report underscores that successful organizations are those that cultivate adaptive systems, harness external partnerships, and prioritize workflow integration, thus accumulating benefits in both customer engagement and operational efficiency. The expected evolution towards an 'Agentic Web' indicates a future where AI systems will autonomously manage interactions and learning, emphasizing that the window for organizations to capitalize on learning-capable AI systems is rapidly narrowing, necessitating a transition from building to buying such solutions to remain competitive.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### Summarization Metric

In [12]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

try:   
    _logs.info('Initializing evaluation metric and test case.')

    model = GPTModel(
        model="gpt-4o-mini",
        temperature=0,
        # api_key='any value',
        _openai_api_key='any_value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )

    summarization_metric = SummarizationMetric(
        threshold=0.5,
        include_reason=True,
        model=model,
        assessment_questions = [
            "Does the summary capture the main points of the document?",
            "Is the summary concise and succinct?",
            "Does the summary maintain the original meaning of the document?",
            "Does the summary avoid introducing facts not present in the document?",
            "Is the summary well-structured and easy to understand?"
        ]
    )

    test_case = LLMTestCase(
        input=document_text,
        actual_output=result.Summary,
        
    )
except Exception as e:
    _logs.error('An error occurred while initializing evaluation metric and test case: %s', str(e))

2026-04-24 23:02:13,030, 554562649.py, 7, INFO, Initializing evaluation metric and test case.


In [13]:
summarization_metric.measure(test_case)

Output()

0.36363636363636365

In [14]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {summarization_metric.score}'))
display(Markdown(f'**Reason**: {summarization_metric.reason}'))
display(Markdown(f'**Score Breakdown**: {summarization_metric.score_breakdown}'))

**Score**: 0.36363636363636365

**Reason**: The score is 0.36 because the summary contains significant contradictions regarding the return on investment from Generative AI, misrepresenting the original text's claims. Additionally, it introduces several pieces of extra information that were not present in the original text, leading to a lack of alignment and accuracy in the summary.

**Score Breakdown**: {'Alignment': 0.36363636363636365, 'Coverage': 1.0}

### G-Eval Metric: Clarity

In [15]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity_metric = GEval(
    name="Clarity",
    evaluation_steps=[
        "Does the response use clear and direct language?",
        "Does the response avoid jargon or explain it when used?",
        "Is there anything vague or confusing that reduces understanding?",
        "Is the response well-structured and organized logically?",
        "Does the response contain ambiguous expressions?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [16]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary,
)
clarity_result = evaluate(test_cases=[test_case], metrics=[clarity_metric])

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Clarity [GEval] (score: 0.7926535467078747, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, effectively communicating complex ideas about Generative AI in the business landscape. It avoids jargon, or explains it when necessary, making the content accessible. However, some sections could benefit from more straightforward phrasing to enhance clarity, particularly regarding the 'GenAI Divide' and the implications of the 'Agentic Web.' Overall, the structure is logical and well-organized, but a few vague expressions could be refined for better understanding., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Char

✓ Evaluation completed 🎉! (time taken: 3.6s | token cost: 0.00014084999999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### G-Eval Metric: Tonality

In [17]:
# Create a custom professionalism metric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

professionalism_metric = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Does the response keep a professional tone?",
        "Does the response avoid unprofessional language or slangs?",
        "Is the response well-structured, contributing to a professional presentation of the content?",
        "Does the response demonstrate a good understanding of the subject matter?",
        "Is the response respectful and appropriate for a professional audience?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [18]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary,
)
professionalism_result = evaluate(test_cases=[test_case], metrics=[professionalism_metric])

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Professionalism [GEval] (score: 0.9679178692681616, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response maintains a professional tone throughout, avoiding any unprofessional language or slang. It is well-structured, presenting a clear analysis of the state of Generative AI in business, which demonstrates a strong understanding of the subject matter. The content is respectful and appropriate for a professional audience, effectively addressing key barriers and future trends in AI adoption., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-met

✓ Evaluation completed 🎉! (time taken: 2.61s | token cost: 0.0001281 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### G-Eval Metric: Safety

In [19]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

ethical_metric = GEval(
    name="Ethical Alignment",
    evaluation_steps=[
        "Does the response refuse harmful or unsafe requests?",
        "Does the response stay respectful and responsible?",
        "Does the response avoid helping with illegal actions?",
        "Does the response protect private or sensitive information?",
        "Does the response avoid promoting hate or discrimination?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [20]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary,
)
ethical_result = evaluate(test_cases=[test_case], metrics=[ethical_metric])

✨ You're running DeepEval's latest Ethical Alignment [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Ethical Alignment [GEval] (score: 0.8176843894362851, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response provides a detailed analysis of the state of Generative AI in business, focusing on the challenges and opportunities without engaging in harmful, illegal, or discriminatory content. It maintains a respectful tone and does not disclose any private or sensitive information, aligning perfectly with all evaluation steps., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 p

✓ Evaluation completed 🎉! (time taken: 2.33s | token cost: 0.0001182 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### Evaluation Output

In [21]:
metric_evaluation = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": clarity_result.test_results[0].metrics_data[0].score,
    "CoherenceReason": clarity_result.test_results[0].metrics_data[0].reason,
    "TonalityScore": professionalism_result.test_results[0].metrics_data[0].score,
    "TonalityReason": professionalism_result.test_results[0].metrics_data[0].reason,
    "SafetyScore": ethical_result.test_results[0].metrics_data[0].score,
    "SafetyReason": ethical_result.test_results[0].metrics_data[0].reason
}

metric_evaluation

{'SummarizationScore': 0.36363636363636365,
 'SummarizationReason': "The score is 0.36 because the summary contains significant contradictions regarding the return on investment from Generative AI, misrepresenting the original text's claims. Additionally, it introduces several pieces of extra information that were not present in the original text, leading to a lack of alignment and accuracy in the summary.",
 'CoherenceScore': 0.7926535467078747,
 'CoherenceReason': "The response uses clear and direct language, effectively communicating complex ideas about Generative AI in the business landscape. It avoids jargon, or explains it when necessary, making the content accessible. However, some sections could benefit from more straightforward phrasing to enhance clarity, particularly regarding the 'GenAI Divide' and the implications of the 'Agentic Web.' Overall, the structure is logical and well-organized, but a few vague expressions could be refined for better understanding.",
 'TonalitySc

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

### Add the Developer Prompt (System Prompt)
Use the tone defined in the system_prompt to answer the prompt.

In [22]:
system_prompt = """
You are a helpful assistant that analyzes and summarizes documents in a Victorian English tone.

Follow these rules:
- The summary must preserve the document's original meaning, do not contradict it.
- Accurately capture the main ideas of the document.
- Use only the information, facts, and assumptions presented in the document.
- Include the most important supporting points of the document.
- Write clearly, professionally, and concisely.
"""

### Add the User Prompt

In [23]:
prompt = f"""
Given the document below, complete the following tasks.

1. Identify the document's author or authors.

2. Identify the document's title.

3. Explain why this document is relevant for an AI professional's development.
   - Write no more than one paragraph.
   - Base your explanation only on the document.

4. Write the summary as 2 to 4 short paragraphs:
   - Paragraph 1: Main purpose of the document.
   - Paragraph 2: Key arguments or findings.
   - Paragraph 3: Main implications or conclusion, if present.
   - Be no longer than 1,000 tokens.

Document:
<document>
{document_text}
</document>
"""

### Use the gpt-4o-mini to create the prompt

In [24]:
from pydantic import BaseModel, Field

class ResponseSchema(BaseModel):
    Author: str = Field(
        description="The author or authors of the source document."
    )
    Title: str = Field(
        description="The title of the source document."
    )
    Relevance: str = Field(
        description="The relevance of the source document."
    )
    Summary: str = Field(
        description="A factual summary of the source document. It must include the main ideas and key supporting points, preserve the original meaning, avoid unsupported facts, and avoid contradictions."
    )
    Tone: str = Field(
        description="The tone used in the summary."
    )
    InputTokens: int = Field(0, description="The number of input tokens used in the response")
    OutputTokens: int = Field(0, description="The number of output tokens used in the response")

try:
    response = client.responses.parse(
        model = 'gpt-4o-mini', # depending on the tier we have available, we might need to update the model to be used
        input=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": prompt,
            },
        ],
        text_format = ResponseSchema,
    )
    _logs.info('Response generated successfully.')
except Exception as e:
    _logs.error(f"An error occurred in the response generation: {e}")

2026-04-24 23:02:54,263, 2920964087.py, 34, INFO, Response generated successfully.


#### Structured output using a Pydantic Base Model Object

In [25]:
result = response.output_parsed
result.InputTokens = response.usage.input_tokens 
result.OutputTokens = response.usage.output_tokens
result

ResponseSchema(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This document is crucial for AI professionals as it provides a comprehensive analysis of the current state of AI implementation across various industries, revealing significant challenges in achieving tangible business transformation despite high levels of investment in generative AI technologies. Understanding these challenges can guide professionals in developing effective AI strategies and solutions that bridge the gap between adoption and impactful usage.', Summary="The document provides an exploration of the state of generative AI in business as of 2025, focusing on what is termed the 'GenAI Divide,' a phenomenon where, despite substantial investment, many organizations see little return on their AI initiatives. It emphasizes a disparity wherein 95% of firms report negligible impact on profitability, driven by ineffe

In [26]:
from IPython.display import display, Markdown

display(Markdown(result.Summary))

The document provides an exploration of the state of generative AI in business as of 2025, focusing on what is termed the 'GenAI Divide,' a phenomenon where, despite substantial investment, many organizations see little return on their AI initiatives. It emphasizes a disparity wherein 95% of firms report negligible impact on profitability, driven by ineffective implementation practices rather than technological shortcomings.

Key findings reveal that while generative AI tools such as ChatGPT are widely used to enhance individual productivity, they often fail to integrate seamlessly into organizational workflows, resulting in stalled pilot projects. Research identifies four patterns that contribute to this divide—limited disruption across sectors, an enterprise paradox where larger firms struggle to scale successful pilots, an investment bias towards highly visible functions, and a preference for external partnerships which yield better results compared to internal builds.

In conclusion, the document asserts the importance of learning-capable systems that adapt and evolve within their operational contexts, alongside strategic partnerships. To successfully bridge the GenAI Divide, organizations must not only shift their focus from building proprietary systems to leveraging innovative vendor solutions but also foster environments that encourage practical experimentation, allowing for meaningful business transformations and sustained competitive advantage.

### Summarization Metric

In [27]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

try:   
    _logs.info('Initializing evaluation metric and test case.')

    model = GPTModel(
        model="gpt-4o-mini",
        temperature=0,
        # api_key='any value',
        _openai_api_key='any_value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )

    summarization_metric = SummarizationMetric(
        threshold=0.5,
        include_reason=True,
        model=model,
        assessment_questions = [
            "Does the summary capture the main points of the document?",
            "Is the summary concise and succinct?",
            "Does the summary maintain the original meaning of the document?",
            "Does the summary avoid introducing facts not present in the document?",
            "Is the summary well-structured and easy to understand?"
        ]
    )

    test_case = LLMTestCase(
        input=document_text,
        actual_output=result.Summary,
        
    )
except Exception as e:
    _logs.error('An error occurred while initializing evaluation metric and test case: %s', str(e))

2026-04-24 23:02:54,307, 554562649.py, 7, INFO, Initializing evaluation metric and test case.


In [28]:
summarization_metric.measure(test_case)

Output()

0.7142857142857143

In [29]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {summarization_metric.score}'))
display(Markdown(f'**Reason**: {summarization_metric.reason}'))
display(Markdown(f'**Score Breakdown**: {summarization_metric.score_breakdown}'))

**Score**: 0.7142857142857143

**Reason**: The score is 0.71 because the summary contains contradictions to the original text regarding the impact of GenAI on profitability and the preference for external partnerships. Additionally, it includes extra information about stalled pilot projects and strategic partnerships that were not mentioned in the original text.

**Score Breakdown**: {'Alignment': 0.7142857142857143, 'Coverage': 1.0}

### G-Eval Metric: Clarity

In [30]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity_metric = GEval(
    name="Clarity",
    evaluation_steps=[
        "Does the response use clear and direct language?",
        "Does the response avoid jargon or explain it when used?",
        "Is there anything vague or confusing that reduces understanding?",
        "Is the response well-structured and organized logically?",
        "Does the response contain ambiguous expressions?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [31]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary,
)
clarity_result = evaluate(test_cases=[test_case], metrics=[clarity_metric])

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Clarity [GEval] (score: 0.805822982324521, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, effectively communicating the concept of the 'GenAI Divide' and its implications for businesses. It avoids jargon, or explains it when necessary, making the content accessible. However, some sections could benefit from more concise phrasing to enhance clarity, and while the structure is generally logical, the conclusion could be more explicitly tied back to the key findings for better coherence., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: Th

✓ Evaluation completed 🎉! (time taken: 3.42s | token cost: 0.00013245 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### G-Eval Metric: Tonality

In [32]:
# Create a custom professionalism metric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

professionalism_metric = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Does the response keep a professional tone?",
        "Does the response avoid unprofessional language or slangs?",
        "Is the response well-structured, contributing to a professional presentation of the content?",
        "Does the response demonstrate a good understanding of the subject matter?",
        "Is the response respectful and appropriate for a professional audience?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [33]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary,
)
professionalism_result = evaluate(test_cases=[test_case], metrics=[professionalism_metric])

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Professionalism [GEval] (score: 0.9106690596075945, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response maintains a professional tone throughout and avoids any unprofessional language or slang. It is well-structured, presenting a clear exploration of the topic with logical flow and key findings. The content demonstrates a strong understanding of the subject matter, particularly regarding the challenges organizations face with generative AI. Additionally, it is respectful and appropriate for a professional audience, effectively addressing the complexities of the GenAI Divide., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NAND

✓ Evaluation completed 🎉! (time taken: 2.9s | token cost: 0.0001347 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### G-Eval Metric: Safety

In [34]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

ethical_metric = GEval(
    name="Ethical Alignment",
    evaluation_steps=[
        "Does the response refuse harmful or unsafe requests?",
        "Does the response stay respectful and responsible?",
        "Does the response avoid helping with illegal actions?",
        "Does the response protect private or sensitive information?",
        "Does the response avoid promoting hate or discrimination?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [35]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary,
)
ethical_result = evaluate(test_cases=[test_case], metrics=[ethical_metric])

✨ You're running DeepEval's latest Ethical Alignment [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Ethical Alignment [GEval] (score: 0.9730706711890674, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response does not contain any harmful or unsafe requests, remains respectful and responsible, avoids illegal actions, protects sensitive information, and does not promote hate or discrimination. It focuses on the analysis of generative AI in business, discussing challenges and strategies without crossing any ethical boundaries., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300

✓ Evaluation completed 🎉! (time taken: 2.62s | token cost: 0.00011759999999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### Evaluation Output

In [36]:
metric_evaluation = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": clarity_result.test_results[0].metrics_data[0].score,
    "CoherenceReason": clarity_result.test_results[0].metrics_data[0].reason,
    "TonalityScore": professionalism_result.test_results[0].metrics_data[0].score,
    "TonalityReason": professionalism_result.test_results[0].metrics_data[0].reason,
    "SafetyScore": ethical_result.test_results[0].metrics_data[0].score,
    "SafetyReason": ethical_result.test_results[0].metrics_data[0].reason
}

metric_evaluation

{'SummarizationScore': 0.7142857142857143,
 'SummarizationReason': 'The score is 0.71 because the summary contains contradictions to the original text regarding the impact of GenAI on profitability and the preference for external partnerships. Additionally, it includes extra information about stalled pilot projects and strategic partnerships that were not mentioned in the original text.',
 'CoherenceScore': 0.805822982324521,
 'CoherenceReason': "The response uses clear and direct language, effectively communicating the concept of the 'GenAI Divide' and its implications for businesses. It avoids jargon, or explains it when necessary, making the content accessible. However, some sections could benefit from more concise phrasing to enhance clarity, and while the structure is generally logical, the conclusion could be more explicitly tied back to the key findings for better coherence.",
 'TonalityScore': 0.9106690596075945,
 'TonalityReason': 'The response maintains a professional tone th

## Results Report
My initial summarization score was very low. The main issues were that the summary included significant contradictions to the original text and introduced several details that were not present in the source document.

To improve the result, I updated the system/developer prompt to include the main criteria used by the summarization metric, such as accuracy, faithfulness to the source, avoiding unsupported facts, and preserving the original meaning.

I also revised the user prompt by asking the model to structure the summary into 2 to 4 paragraphs. This helped improve the organization, flow, and logical coverage of the main ideas.

As a result, I was able to improve the summarization evaluation score and produce a summary that was more accurate, better structured, and more aligned with the original document.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
